# Independent re-derivation of both manuscripts

**What this notebook is.** A second, deliberately independent implementation of the
quantitative content of

* *Spectral kernels and Feynman-diagram weights for screened-exchange energies: Analytic structure of screened second-order exchange in the electron gas* (JPA), and
* *Spin-polarized low-density asymptotics of RPA-screened second-order exchange in the uniform electron gas* (PRB).

**Method.** Nothing here imports `src/ueg_exchange`, `production/`, `spin/reviewer_numerics.py`
or any other module of this package, and no value is copied from the reviewer notebooks.
Every quantity is rebuilt from the definitions printed in the manuscripts, with its own
quadrature, its own Monte-Carlo estimator and its own symbolic algebra, and only then
compared with the printed value. Where a manuscript result can be reached by two routes,
both are computed here.

**How to read it.** Each cell prints `OK` or `FAIL` per item and a count at the end.
A `FAIL` means this notebook disagrees with the manuscript. The final cell aggregates
everything.

**Provenance.** This notebook was produced by an AI assistant (Claude) acting as an
adversarial referee at the author's request, and it was executed in full before
distribution. It is supplementary evidence, not part of the locked production chain:
it deliberately uses ordinary floating point and modest sample sizes so that it runs in
a few minutes on any machine, and its Monte-Carlo errors are therefore larger than those
of `production/`. Sections 1-4 are deterministic and reproduce exactly; Section 5 is
stochastic with fixed seeds.

**Runtime.** Runtime depends on the machine; see the current execution receipt. Restart and Run All.

In [1]:
import numpy as np, sympy as sp, mpmath as mp, platform, time
mp.mp.dps = 22
T0 = time.perf_counter()
LOG = []
def rec(tag, got, ref, tol=1e-10):
    g, r = mp.mpf(got), mp.mpf(ref)
    err = abs(g - r)/max(1, abs(r))
    ok = err < tol
    LOG.append(ok)
    print(('OK   ' if ok else 'FAIL ') + f'{tag:<56} {mp.nstr(g,12):>16}  vs {mp.nstr(r,12):>16}  rel {mp.nstr(err,2)}')
    return ok
def note(tag, ok, detail=''):
    LOG.append(bool(ok))
    print(('OK   ' if ok else 'FAIL ') + f'{tag:<56} {detail}')
print('Python', platform.python_version(), '| NumPy', np.__version__,
      '| SymPy', sp.__version__, '| mpmath', mp.__version__, '| dps', mp.mp.dps)
pi = mp.pi; gam = mp.euler; log2 = mp.log(2)
A0 = 3/(32*pi**5); VB = 4*pi/3
alphaL = (4/(9*pi))**(mp.mpf(1)/3); aTF = 4*alphaL/pi
CG = mp.sqrt(2*pi)/pi**2*mp.gamma(mp.mpf(3)/4)/mp.gamma(mp.mpf(1)/4)

Python 3.12.14 | NumPy 2.3.5 | SymPy 1.14.0 | mpmath 1.3.0 | dps 22


## 1. Normalization, the Onsager constant and the Lorentzian kernel endpoints

`A0` is rebuilt from the factor list printed after JPA Eq. (8). `D_0` is computed twice:
from the closed form and from an independent quadrature of the reduced $u$ integral whose
integrand is written here from its own definition. The endpoint coefficients $a_0,b_0$ are
obtained both from the Weber-Schafheitlin ratio and from a direct oscillatory quadrature
of $j_1^2$.

In [2]:
rec('A0 factor chain = 3/(32 pi^5)',
    2*mp.mpf(1)/2*2*mp.mpf(1)/2*(4*pi)**2*(2*pi)**-9*(3*pi**2), A0)
D0 = pi**3/18*log2 - pi/4*mp.zeta(3)
rec('D0 closed form', D0, mp.mpf('0.2499018971'), tol=1e-10)
L  = lambda u: mp.log((u+1)/(u-1))
h0 = lambda u: -mp.log(u)*(2/(u+1)-mp.log(1+1/u)) + mp.polylog(2,-1/u) + 2*mp.log(1+1/u)
rec('D0 = (pi/3) int_1^inf L(u) h0(u) du  [own quadrature]',
    pi/3*mp.quad(lambda t: h0(1/t)*L(1/t)/t**2, [0, mp.mpf('0.5'), 1]), D0, tol=1e-14)
e2x = 6/pi**3*D0
rec('e2x = (6/pi^3) D0 = ln2/3 - 3 zeta(3)/(2 pi^2)', e2x, log2/3 - 3*mp.zeta(3)/(2*pi**2))
rec('e2x in mRy', 1000*e2x, mp.mpf('48.3583178'), tol=1e-8)

Ij = lambda s: pi/2*mp.gamma(2-s)*mp.gamma(1+s/2)/(2**(2-s)*mp.gamma((3-s)/2)**2*mp.gamma(3-s/2))
rec('a0 = 2 I_j(0) = 1/2', 2*Ij(0), mp.mpf(1)/2)
rec("b0 = 2[(1-log2)I_j(0) - I_j'(0)] = gamma/2 - 3/8",
    2*((1-log2)*Ij(0)-mp.diff(Ij,0)), gam/2 - mp.mpf(3)/8, tol=1e-14)
j1 = lambda x: mp.sin(x)/x**2 - mp.cos(x)/x
rec('a0 from direct oscillatory quadrature of j1^2',
    2*mp.quadosc(lambda t: j1(t/2)**2/t, [0, mp.inf], period=4*pi), mp.mpf(1)/2, tol=1e-8)

OK   A0 factor chain = 3/(32 pi^5)                            0.000306352841536  vs 0.000306352841536  rel 6.5e-27
OK   D0 closed form                                             0.249901897104  vs     0.2499018971  rel 4.3e-12
OK   D0 = (pi/3) int_1^inf L(u) h0(u) du  [own quadrature]      0.249901897104  vs   0.249901897104  rel 5.6e-23
OK   e2x = (6/pi^3) D0 = ln2/3 - 3 zeta(3)/(2 pi^2)            0.0483583178363  vs  0.0483583178363  rel 9.1e-24
OK   e2x in mRy                                                  48.3583178363  vs       48.3583178  rel 7.5e-10
OK   a0 = 2 I_j(0) = 1/2                                                   0.5  vs              0.5  rel 6.6e-24
OK   b0 = 2[(1-log2)I_j(0) - I_j'(0)] = gamma/2 - 3/8         -0.0863921675492  vs -0.0863921675492  rel 3.3e-24
OK   a0 from direct oscillatory quadrature of j1^2                         0.5  vs              0.5  rel 0.0


## 2. The screening kernel, its Mellin transform and the large-$\Lambda$ coefficients

$\Xi$ is evaluated from two representations. $K(s)$ is compared with a direct quadrature
inside the strip. $C_1^\Lambda$ and $C_0^\Lambda$ are then obtained a second time from the
real-integral moments of $\Xi-1$, which is the route of JPA Appendix F and does not use the
residue calculation.

In [3]:
Xi   = lambda z: z*mp.quad(lambda t: mp.e**(-z*t)/(1+t**2), [0, 1/mp.mpf(max(z,1)), 1, mp.inf])
Xim1 = lambda z: -mp.quad(lambda w: mp.e**(-w)*w**2/(z**2+w**2), [0, 1, 10, mp.inf])
for z in ['0.5', '3', '40']:
    zz = mp.mpf(z); rec(f'Xi({z}) - 1: two representations agree', Xi(zz)-1, Xim1(zz), tol=1e-14)
K = lambda s: -pi/2*mp.gamma(s+1)/mp.sin(pi*s/2)
for s in ['-0.3', '-0.7']:
    sv = mp.mpf(s)
    rec(f'K({s}) = int z^(s-1) Xi dz',
        mp.quad(lambda z: z**(sv-1)*Xi(z), [0, mp.mpf('0.1'), 1, 10, 100, 1000, mp.inf]), K(sv), tol=1e-6)
rec('K(1) = -pi/2', K(1), -pi/2)
rec("K'(1) = -pi(1-gamma)/2", mp.diff(K,1), -pi*(1-gam)/2, tol=1e-14)
m0 = mp.quad(Xim1, [0,1,10,100,mp.inf])
m1 = mp.quad(lambda z: mp.log(z)*Xim1(z), [0,1,10,100,1000,mp.inf])
rec('int [Xi-1] dz = -pi/2', m0, -pi/2, tol=1e-8)
rec('int log z [Xi-1] dz = -pi(1-gamma)/2', m1, -pi*(1-gam)/2, tol=1e-6)
a0, b0 = mp.mpf(1)/2, gam/2 - mp.mpf(3)/8
rec('C_1^Lambda from moments = pi/(4 D0)', -a0*m0/D0, pi/(4*D0), tol=1e-8)
rec('C_1^Lambda printed value', pi/(4*D0), mp.mpf('3.14282593'), tol=1e-8)
rec('C_0^Lambda from moments = -C_1/4', (a0*m1+b0*m0)/D0, -pi/(16*D0), tol=1e-6)

OK   Xi(0.5) - 1: two representations agree                    -0.569736617137  vs  -0.569736617137  rel 0.0
OK   Xi(3) - 1: two representations agree                      -0.124126867924  vs  -0.124126867924  rel 3.3e-24
OK   Xi(40) - 1: two representations agree                    -0.00124079494926  vs -0.00124079494926  rel 9.4e-24
OK   K(-0.3) = int z^(s-1) Xi dz                                  4.4912405217  vs    4.49124056488  rel 9.6e-9
OK   K(-0.7) = int z^(s-1) Xi dz                                 5.27397434177  vs    5.27397437577  rel 6.4e-9
OK   K(1) = -pi/2                                               -1.57079632679  vs   -1.57079632679  rel 0.0
OK   K'(1) = -pi(1-gamma)/2                                    -0.664108080599  vs  -0.664108080599  rel 0.0
OK   int [Xi-1] dz = -pi/2                                      -1.57079632679  vs   -1.57079632679  rel 1.1e-19
OK   int log z [Xi-1] dz = -pi(1-gamma)/2                      -0.664108080599  vs  -0.664108080599  rel 5.4

## 3. Geometry of both papers from the lens representation

$I_0$, $I_2$ and $\int C\,\mathrm dq$ are rebuilt from the overlap volume of two unit balls.
For $I_2$ the split $\bm d=\bm p-\bm k$, $\bm s=(\bm p+\bm k)/2$ turns the integrand into
$3s_z^2+d_z^2/4$, which is integrated here without using the manuscript's value.
$A_2$ follows exactly.

In [4]:
Vcl = lambda r: pi*(4+r)*(2-r)**2/12
rec('overlap volume V(1) = pi(4+r)(2-r)^2/12',
    2*pi*mp.quad(lambda s: 1-(s+mp.mpf(1)/2)**2, [0, mp.mpf(1)/2]), Vcl(1))
rec('I_0 = int_{BxB} |p-k|^-2 = 4 pi^2', 4*pi*mp.quad(Vcl, [0,2]), 4*pi**2)
def i2(r):
    h = 1-r/2
    A = 2*pi*mp.quad(lambda s: s**2*(1-(s+r/2)**2), [0,h])
    B = 2*pi*mp.quad(lambda s: (1-(s+r/2)**2)**2/4, [0,h])
    return 4*pi*A + 8*pi*B + pi*r**2*Vcl(r)/3
rec('I_2 = int (p_z^2+k_z^2+p_z k_z)/|p-k|^2 = 52 pi^2/27',
    mp.quad(i2,[0,1,2]), 52*pi**2/27, tol=1e-9)
rec('104 pi^2/27 = 2 I_2', 104*pi**2/27, 2*52*pi**2/27)
def Cint():
    def f(r,c):
        s = mp.sqrt(1-c**2)
        return 0 if s < mp.mpf('1e-12') else Vcl(r)*2*pi*r**2*(pi/2-mp.atan(c/s))/(r*s)
    return mp.quad(lambda r: mp.quad(lambda c: f(r,c), [-1,0,1]), [0,1,2])
IC = Cint()
rec('int_0^inf C(q) dq = 8 pi^4/15', IC, 8*pi**4/15, tol=1e-8)
rec('A_2 = 4 pi A0 (-4/3) Fp int Z dq = 8/15',
    4*pi*A0*(-mp.mpf(4)/3)*(-2*IC), mp.mpf(8)/15, tol=1e-8)
rec('C_inf = (1/16pi) 2 (4pi/3)^2 = 2 pi/9', (1/(16*pi))*2*VB**2, 2*pi/9)

OK   overlap volume V(1) = pi(4+r)(2-r)^2/12                       1.308996939  vs      1.308996939  rel 2.0e-23
OK   I_0 = int_{BxB} |p-k|^-2 = 4 pi^2                           39.4784176044  vs    39.4784176044  rel 0.0
OK   I_2 = int (p_z^2+k_z^2+p_z k_z)/|p-k|^2 = 52 pi^2/27        19.0081269947  vs    19.0081269947  rel 2.2e-23
OK   104 pi^2/27 = 2 I_2                                         38.0162539894  vs    38.0162539894  rel 0.0
OK   int_0^inf C(q) dq = 8 pi^4/15                               51.9515152181  vs    51.9515152181  rel 6.3e-13
OK   A_2 = 4 pi A0 (-4/3) Fp int Z dq = 8/15                    0.533333333333  vs   0.533333333333  rel 3.3e-13
OK   C_inf = (1/16pi) 2 (4pi/3)^2 = 2 pi/9                      0.698131700798  vs   0.698131700798  rel 1.3e-23


## 4. Frequency profiles and the three third-order pieces

$R(c)=2\int_0^1\lambda\,\mathrm d\lambda/(1+\lambda c)$ is used in the form that has no
cancellation. $F$ and $J_C$ are checked against their closed forms by quadrature.
For $\Phi_4$ and $\Psi$ this notebook derives its own coupling-first reduction,
$\Phi_4(g)=2\int_0^1\lambda\,\mathcal R_4(\lambda g)\,\mathrm d\lambda$ and
$\Psi(g)=-2\int_0^1\lambda^2\mathcal R_L(\lambda g)\,\mathrm d\lambda$ with the residues
$\mathcal R_4,\mathcal R_L$ printed in PRB Appendix A, checks the reduction against the
direct double quadrature, and then integrates it. $A_1,A_2,A_3$ come out of these.

In [5]:
def R(c):
    c = mp.mpf(c)
    return 2*(c-mp.log(1+c))/c**2 if c > mp.mpf('1e-3') else mp.nsum(lambda n: 2*(-c)**int(n)/(int(n)+2), [0,30])
ReA = lambda u,n: mp.re((1j*u - mp.mpf(1)/2)**(-n))
Fcl = lambda g: -mp.mpf(4)/3*(mp.sqrt(1+g)-1)/(mp.sqrt(1+g)+1)**2
Fq  = lambda g: mp.quad(lambda u: (R(g/(1+4*u**2))-1)*(mp.mpf(1)/4-u**2)/(mp.mpf(1)/4+u**2)**2/pi, [0,1,10,mp.inf])
for g in ['0.2','3']:
    rec(f'F({g}): quadrature vs closed form', Fq(mp.mpf(g)), Fcl(mp.mpf(g)), tol=1e-12)
JCc = lambda g: (mp.mpf(4)/(3*g**2)*((1+g)**mp.mpf(1.5)-3*mp.sqrt(1+g)+2) if g > mp.mpf('1e-6')
                else 1-g/3+3*g**2/16-g**3/8)   # Taylor series of the closed form
for g in ['0.4','5']:
    gg = mp.mpf(g)
    rec(f'J_C({g}): coupling integral vs closed form',
        2*mp.quad(lambda l: l/mp.sqrt(1+l*gg), [0,1]), JCc(gg), tol=1e-14)
A1 = 3/pi**2*mp.quad(lambda x: Fcl(x**-4)/x**2, [0, mp.mpf('0.5'), 1, 2, mp.inf])
A1c = -4/(7*mp.sqrt(2)*pi**mp.mpf(1.5))*mp.gamma(mp.mpf(1)/4)/mp.gamma(mp.mpf(3)/4)
rec('A_1 = (3/pi^2) int F(x^-4)/x^2 dx vs closed form', A1, A1c, tol=1e-12)
rec('A_1 numerical value', A1c, mp.mpf('-0.21469376098'), tol=1e-10)
eps = mp.mpf('1e-4')
Fp = mp.quad(lambda x: x**-4*JCc(x**-4), [eps, mp.mpf('0.3'), 1, 3, mp.inf]) - 4/(3*eps)
A3c = 8*pi*A0*VB**2*Fp
rec('A_{3,cross} = -(32/15) C_Gamma', A3c, -mp.mpf(32)/15*CG, tol=1e-6)
res4 = lambda m: -4*m/(mp.sqrt(1+m)*(mp.sqrt(1+m)+1)**4)
resL = lambda m: (lambda s: 8*(2*s**3-s**2+4*s+1)/(15*s**3*(s+1)**4))(mp.sqrt(1+m))
Phi4 = lambda g: 2*mp.quad(lambda l: l*res4(l*g), [0,1])
Psi  = lambda g: -2*mp.quad(lambda l: l**2*resL(l*g), [0,1])
lindhard_coeff_4 = lambda u: -16*(12*u**2-1)/(15*(1+4*u**2)**3)
def Rp(c):
    c = mp.mpf(c)
    return 2*(c**2/(1+c)-2*(c-mp.log(1+c)))/c**3 if c > mp.mpf('1e-3') else mp.nsum(lambda n: 2*int(n)*(-1)**int(n)*c**(int(n)-1)/(int(n)+2), [1,30])
gg = mp.mpf('0.8')
rec('Phi_4(0.8): own coupling-first reduction vs double quadrature', Phi4(gg),
    mp.quad(lambda u: R(gg/(1+4*u**2))*ReA(u,4)/pi, [0,1,10,mp.inf]), tol=1e-12)
rec('Psi(0.8): own coupling-first reduction vs double quadrature', Psi(gg),
    mp.quad(lambda u: Rp(gg/(1+4*u**2))*lindhard_coeff_4(u)*ReA(u,2)/pi, [0,1,10,mp.inf]), tol=1e-12)
A3k = 4*pi*A0*(104*pi**2/27)*mp.quad(lambda x: x**-4*Phi4(x**-4), [0, mp.mpf('0.4'), 1, 3, 10, mp.inf])
A3L = 24*pi**3*A0*mp.quad(lambda x: x**-8*Psi(x**-4), [0, mp.mpf('0.4'), 1, 3, 10, mp.inf])
rec('A_{3,kin} = -(416/675) C_Gamma', A3k, -mp.mpf(416)/675*CG, tol=1e-8)
rec('A_{3,L}   = -(264/125) C_Gamma', A3L, -mp.mpf(264)/125*CG, tol=1e-8)
rec('A_3 total = -(16408/3375) C_Gamma', A3c+A3k+A3L, -mp.mpf(16408)/3375*CG, tol=1e-6)
rec('A_3 numerical value', -mp.mpf(16408)/3375*CG, mp.mpf('-0.41732536262'), tol=1e-10)
for n,(A,p) in enumerate([(A1c,mp.mpf('-0.2213782293')),(mp.mpf(8)/15,mp.mpf('0.5670608947')),
                          (-mp.mpf(16408)/3375*CG,mp.mpf('-0.4575317487'))], start=1):
    rec(f'printed r_s coefficient {n}', A/(4*aTF/3)**(mp.mpf(n)/4), p, tol=1e-10)

OK   F(0.2): quadrature vs closed form                        -0.0289827681132  vs -0.0289827681132  rel 2.5e-24
OK   F(3): quadrature vs closed form                           -0.148148148148  vs  -0.148148148148  rel 3.3e-24
OK   J_C(0.4): coupling integral vs closed form                 0.890453911734  vs   0.890453911734  rel 3.0e-22
OK   J_C(5): coupling integral vs closed form                   0.498585025512  vs   0.498585025512  rel 6.6e-24
OK   A_1 = (3/pi^2) int F(x^-4)/x^2 dx vs closed form          -0.214693760989  vs  -0.214693760989  rel 6.6e-24
OK   A_1 numerical value                                       -0.214693760989  vs   -0.21469376098  rel 9.1e-12
OK   A_{3,cross} = -(32/15) C_Gamma                             -0.18312668277  vs   -0.18312668277  rel 9.0e-14
OK   Phi_4(0.8): own coupling-first reduction vs double quadrature -0.0654519597969  vs -0.0654519597969  rel 0.0
OK   Psi(0.8): own coupling-first reduction vs double quadrature -0.0583882402049  vs -0.058388

## 5. Spin structure and the exact full-polarization identity

$\mathcal S_3$ is required to be exactly the combination that the $q_*^{-3}$ split
$(1856/675,\,264/125)$ produces; this is checked as an identity in $\zeta$, not at a point.
The full-polarization rescaling is then applied to the three-term expansion itself.

In [6]:
z = sp.symbols('zeta')
xs = [(1+s*z)**sp.Rational(1,3) for s in (1,-1)]
M  = lambda n: sum(x**n for x in xs)/2
rec('M_3(zeta) = 1 identically', 0 if sp.simplify(M(3)-1) == 0 else 1, 0)
for n,v in [(4,2**sp.Rational(1,3)),(5,2**sp.Rational(2,3)),(6,2)]:
    rec(f'M_{n}(1) = 2^({n}/3-1)', mp.mpf(str(sp.N(M(n).subs(z,1),22))), mp.mpf(str(sp.N(v,22))))
S3 = (1160*M(6)+891*M(4)*M(5))/2051
S3_split = (sp.Rational(1856,675)*M(6)+sp.Rational(264,125)*M(4)*M(5))/sp.Rational(16408,3375)
note('S_3 equals the (1856/675, 264/125) split identically in zeta',
     sp.simplify(S3-S3_split) == 0)
rec('S_3(0) = 1', mp.mpf(str(sp.N(S3.subs(z,0),22))), 1)
rec('S_3(1) = 2', mp.mpf(str(sp.N(S3.subs(z,1),22))), 2)
c1,c2,c3 = mp.mpf('-0.2213782293'), mp.mpf('0.5670608947'), mp.mpf('-0.4575317487')
eps = lambda rs,zz: (c1*mp.mpf(str(sp.N(M(4).subs(z,zz),22)))*rs**mp.mpf('-0.25')
                   + c2*mp.mpf(str(sp.N(M(5).subs(z,zz),22)))*rs**mp.mpf('-0.5')
                   + c3*mp.mpf(str(sp.N(S3.subs(z,zz),22)))*rs**mp.mpf('-0.75'))
for rs in [30, 10000]:
    rec(f'eps(rs={rs}, zeta=1) = eps(2^(-4/3) rs, 0)', eps(mp.mpf(rs),1), eps(2**mp.mpf('-4/3')*rs,0), tol=1e-18)

OK   M_3(zeta) = 1 identically                                             0.0  vs              0.0  rel 0.0
OK   M_4(1) = 2^(4/3-1)                                          1.25992104989  vs    1.25992104989  rel 0.0
OK   M_5(1) = 2^(5/3-1)                                          1.58740105197  vs    1.58740105197  rel 0.0
OK   M_6(1) = 2^(6/3-1)                                                    2.0  vs              2.0  rel 0.0
OK   S_3 equals the (1856/675, 264/125) split identically in zeta 
OK   S_3(0) = 1                                                            1.0  vs              1.0  rel 0.0
OK   S_3(1) = 2                                                            2.0  vs              2.0  rel 0.0
OK   eps(rs=30, zeta=1) = eps(2^(-4/3) rs, 0)                 -0.0262193380292  vs -0.0262193380292  rel 5.0e-23
OK   eps(rs=10000, zeta=1) = eps(2^(-4/3) rs, 0)               -0.019805441998  vs  -0.019805441998  rel 6.6e-24


## 6. The JPA static reduction, checked symbolically and by its sum rule

The 4D form of JPA Appendix D rests on the identity
$A^2-4RS=4(x+y)^2H$ with $A=R+S+(x+y)^2$. That is verified here as an exact polynomial
identity. $X_c(q)$ is then evaluated from that form for $q<2$ and from a second
representation derived here for $q>2$, and the sum rule $\int_0^\infty X_c\,\mathrm dq=D_0$
is a strong test of the whole normalization.

In [7]:
q,t,s_,x,y = sp.symbols('q t s x y', positive=True)
Rr = 1-x**2-q**2/4+q*x*t; Ss = 1-y**2-q**2/4+q*y*s_
Aa = Rr+Ss+(x+y)**2
Hh = 1+q*x*y*(t+s_)/(x+y)+q**2/4*((x*t-y*s_)**2/(x+y)**2-1)
note('A^2 - 4RS - 4(x+y)^2 H = 0 exactly',
     sp.simplify(sp.expand(Aa**2-4*Rr*Ss-4*(x+y)**2*Hh)) == 0)

def gl(n,lo,hi):
    a,b = np.polynomial.legendre.leggauss(n); return 0.5*(hi-lo)*a+0.5*(hi+lo), 0.5*(hi-lo)*b
gx,gw = np.polynomial.legendre.leggauss(44)
def Xc_small(qv):
    tot = 0.0
    for ti,wi in zip(gx,gw):
        Lt = qv*ti/2+np.sqrt(1-qv**2*(1-ti**2)/4)
        for sj,wj in zip(gx,gw):
            Ls = qv*sj/2+np.sqrt(1-qv**2*(1-sj**2)/4)
            xx = 0.5*Lt*(gx+1); wx = 0.5*Lt*gw
            yy = 0.5*Ls*(gx+1); wy = 0.5*Ls*gw
            X,Y = np.meshgrid(xx,yy,indexing='ij')
            H = 1+qv*X*Y*(ti+sj)/(X+Y)+qv**2/4*((X*ti-Y*sj)**2/(X+Y)**2-1)
            tot += wi*wj*(wx[:,None]*wy[None,:]*(X*Y/(X+Y)**2/np.sqrt(H))).sum()
    return float(np.pi/16*tot*qv)
Vov = lambda r: np.where(r < 2, np.pi*(4+r)*(2-r)**2/12, 0.0)
def Xc_large(qv, n=300):
    # q >= 2: the two crescents are whole balls, so X_c reduces to a 2D integral
    # over d = p - k with the static energy denominator q(d_z + q) kept explicit.
    rr,wr = gl(n,0.0,2.0); cc,wc = gl(n,-1.0,1.0)
    Rm,Cm = np.meshgrid(rr,cc,indexing='ij')
    V = Vov(rr)[:,None]
    dz = Rm*Cm
    dens = 2*np.pi*Rm**2*V/((qv*dz+qv*qv)*((dz+qv)**2+Rm**2*(1-Cm**2)))
    return float((wr[:,None]*wc[None,:]*dens).sum()/(8*np.pi))
c0cl = float(pi/8*(2*log2-1)); c2cl = float(pi/1152-pi*log2/96)
rec('X_c(q)/q -> c_0 = (pi/8)(2 log2 - 1)', Xc_small(0.02)/0.02, c0cl, tol=1e-4)
rec('(X_c/q - c_0)/q^2 -> c_2 = pi/1152 - pi log2/96',
    (Xc_small(0.05)/0.05-c0cl)/0.05**2, c2cl, tol=2e-3)
rec('q^4 X_c(q) -> C_inf = 2 pi/9', 60.0**4*Xc_large(60.0), float(2*pi/9), tol=2e-3)
qa,wa = gl(90,0.0,2.0)
tt,wt = gl(140,0.0,1.0); qb = 2+2*tt/(1-tt); jb = wt*2/(1-tt)**2
sumrule = sum(w*Xc_small(qq) for qq,w in zip(qa,wa)) + sum(w*Xc_large(qq) for qq,w in zip(qb,jb))
rec('sum rule  int_0^inf X_c dq = D_0', sumrule, D0, tol=1e-3)
A34 = pi**2/(9*mp.sqrt(2))*mp.mpf('0.75')**mp.mpf('0.75')*(pi/(4*alphaL))**mp.mpf('0.75')
rec('A_{3/4} rederived from C_inf and f_L ~ 4/(3q^2)',
    (2*pi/9)*pi/(2*mp.sqrt(2))*(mp.mpf(3)/4)**mp.mpf('0.75')*(pi/(4*alphaL))**mp.mpf('0.75'), A34)
rec('A_{3/4} printed value', A34, mp.mpf('0.8501'), tol=1e-4)
rec('c_log = 2 alpha_L c_0/pi = alpha_L(2 log2 - 1)/4',
    2*alphaL/pi*(pi/8*(2*log2-1)), alphaL*(2*log2-1)/4)
calA = 4*pi**2*(1-log2)/3
rec('calA = 4 pi int h^2 dz = 4 pi^2 (1-log2)/3',
    4*pi*mp.quad(lambda zz: (1-zz*mp.atan(1/zz))**2, [0,1,10,mp.inf]), calA, tol=1e-12)
rec('C_log2 = -(pi A0/3) a_TF calA', -(pi*A0/3)*aTF*calA, -alphaL*(1-log2)/(6*pi**3))
rec('C_log2 printed value (Ry)', -alphaL*(1-log2)/(6*pi**3), mp.mpf('-0.000859'), tol=1e-6)

OK   A^2 - 4RS - 4(x+y)^2 H = 0 exactly                       
OK   X_c(q)/q -> c_0 = (pi/8)(2 log2 - 1)                       0.151689580093  vs   0.151697440877  rel 7.9e-6
OK   (X_c/q - c_0)/q^2 -> c_2 = pi/1152 - pi log2/96           -0.019909649606  vs -0.0199561114844  rel 4.6e-5
OK   q^4 X_c(q) -> C_inf = 2 pi/9                               0.698442151576  vs   0.698131700798  rel 0.00031
OK   sum rule  int_0^inf X_c dq = D_0                           0.249902198541  vs   0.249901897104  rel 3.0e-7
OK   A_{3/4} rederived from C_inf and f_L ~ 4/(3q^2)            0.850136562028  vs   0.850136562028  rel 0.0
OK   A_{3/4} printed value                                      0.850136562028  vs           0.8501  rel 3.7e-5
OK   c_log = 2 alpha_L c_0/pi = alpha_L(2 log2 - 1)/4          0.0503208050365  vs  0.0503208050365  rel 8.3e-25
OK   calA = 4 pi int h^2 dz = 4 pi^2 (1-log2)/3                  4.03802124964  vs    4.03802124964  rel 0.0
OK   C_log2 = -(pi A0/3) a_TF calA           

## 7. An independent Monte-Carlo evaluation of the kite energy

The estimator below is written from the definition of $\mathcal K[f]$ only. It draws
$\bm p$ from a mixture of the two unit balls, so its density on the support of
$\Delta_p$ is $1/(2V_B)$, and draws $\bm d=\bm p-\bm k$ with radius uniform in $[0,2+q]$,
whose density $[4\pi(2+q)d^2]^{-1}$ cancels the bare Coulomb singularity. The occupation
signs are applied as they stand.

The first test is the one that can falsify the whole convention chain: with the bare line
the estimator must return the Onsager-Mittag-Stephen constant $e_{2x}$.

Errors here are replicate standard errors. Per-frequency errors from a shared geometry
sample are correlated, so summing them in quadrature would understate the error.

In [8]:
def sample(q, N, rng):
    which = rng.integers(0, 2, N)
    u = rng.normal(size=(N,3)); u /= np.linalg.norm(u, axis=1)[:,None]
    p = u*(rng.random(N)**(1/3))[:,None]
    p[which == 1, 2] -= q
    inB  = np.einsum('ij,ij->i', p, p) <= 1.0
    pq = p.copy(); pq[:,2] += q
    inBq = np.einsum('ij,ij->i', pq, pq) <= 1.0
    Dp = inB.astype(float) - inBq.astype(float)
    ud = rng.normal(size=(N,3)); ud /= np.linalg.norm(ud, axis=1)[:,None]
    d = ud*(rng.random(N)*(2.0+q))[:,None]
    k = p-d
    inK  = np.einsum('ij,ij->i', k, k) <= 1.0
    kq = k.copy(); kq[:,2] += q
    inKq = np.einsum('ij,ij->i', kq, kq) <= 1.0
    Dk = inK.astype(float) - inKq.astype(float)
    w = Dp*Dk*(~(inB & inBq))
    return w, p[:,2]*q+q*q/2, k[:,2]*q+q*q/2, 2*float(VB)*4*np.pi*(2.0+q)

def Xq(q, nus, N, rng):
    w,a,b,norm = sample(q, N, rng)
    nz = w != 0; w,a,b = w[nz],a[nz],b[nz]
    out = np.empty(len(nus))
    for i,nu in enumerate(nus):
        num = a*b-nu*nu
        out[i] = norm*(w*num/(num*num+nu*nu*(a+b)**2)).sum()/N
    return out

def lindhard_shape(q, nu, n=160):
    k,w = gl(n, 0.0, 1.0)
    return (w*k*np.log(((q*k+q*q/2)**2+nu**2)/((q*k-q*q/2)**2+nu**2))).sum()/(2*q)

def Rnp(c):
    c = np.asarray(c, float); out = np.empty_like(c); sm = c < 1e-3
    cs = c[sm]; out[sm] = 1-2*cs/3+cs**2/2-2*cs**3/5
    cb = c[~sm]; out[~sm] = 2*(cb-np.log1p(cb))/cb**2
    return out

E2X = float(e2x); ATF = float(aTF)
def kite(rs, breaks, nsegs, nnu=40, N=200_000, seed=1, bare=False):
    rng = np.random.default_rng(seed); q_tf_sq = ATF*rs; tot = 0.0
    nodes = []
    for i,n in enumerate(nsegs):
        lo,hi = breaks[i], breaks[i+1]
        if hi is None:
            tq,wq = gl(n,0.0,1.0); nodes += list(zip(lo+2*tq/(1-tq), wq*2/(1-tq)**2))
        else:
            qq,wq = gl(n,lo,hi); nodes += list(zip(qq,wq))
    for qv,wv in nodes:
        xs = qv+qv*qv/2
        sg,wsg = gl(nnu,0.0,1.0); nus = xs*sg/(1-sg); jx = xs/(1-sg)**2
        X = Xq(qv, nus, N, rng)
        f = np.ones_like(nus) if bare else Rnp(np.array([q_tf_sq/qv**2*lindhard_shape(qv,xx) for xx in nus]))-1.0
        tot += wv*4*np.pi*float(A0)*(wsg*jx*f*X).sum()/np.pi
    return tot if bare else E2X+tot

t = time.perf_counter()
v = [1000*kite(1.0, (0.0,1.0,4.0,None), (14,14,22), N=300_000, seed=40+i, bare=True) for i in range(6)]
m, se = np.mean(v), np.std(v,ddof=1)/np.sqrt(6)
note('bare limit K[v] = e_2x (Onsager-Mittag-Stephen)', abs(m-1000*E2X) < 3*se,
     f'{m:.3f} +- {se:.3f} mRy vs exact {1000*E2X:.3f}   ({(m-1000*E2X)/se:+.1f} sigma, {time.perf_counter()-t:.0f}s)')

OK   bare limit K[v] = e_2x (Onsager-Mittag-Stephen)          48.236 +- 0.112 mRy vs exact 48.358   (-1.1 sigma, 22s)


### 7.1 Metallic densities against PRB Tables II and III

The published columns are quoted here only for orientation; the comparison that matters is
against the direct evaluation of the manuscript.

In [9]:
PRBDIRECT = {1.0: (27.828,0.045), 5.0: (6.715,0.076)}
for rs in [1.0, 5.0]:
    st = np.sqrt(ATF*rs); qs = max(1.0,(4*ATF*rs/3)**0.25)
    v = [1000*kite(rs, (0.0,min(st,0.5),max(2*qs,2.0),None), (14,26,20), N=200_000, seed=70+i) for i in range(6)]
    m, se = np.mean(v), np.std(v,ddof=1)/np.sqrt(6)
    p,pe = PRBDIRECT[rs]
    note(f'kite energy at r_s={rs:g}', abs(m-p) < 3*np.hypot(se,pe),
         f'{m:7.3f} +- {se:5.3f} mRy   manuscript direct {p:7.3f}({pe:.3f})   {(m-p)/np.hypot(se,pe):+.1f} sigma')

OK   kite energy at r_s=1                                      27.849 +- 0.012 mRy   manuscript direct  27.828(0.045)   +0.4 sigma
OK   kite energy at r_s=5                                       6.761 +- 0.047 mRy   manuscript direct   6.715(0.076)   +0.5 sigma


### 7.2 Strong screening against the three-term expansion

This compares the original integral with the sum of the three-term expansion; it does not
separate the coefficients. At $r_s=10^8$ the second term is $0.0567$ mRy and the third
$-0.00046$ mRy, both below the Monte Carlo standard error ($\approx0.13$ mRy); $A_1$, $A_2$ and
$A_3$ are tested individually by the analytic and quadrature checks of the other notebooks. The
scale is $q_*\simeq97$, and the $q$ grid must resolve both $q\sim1$ and $q\sim q_*$; a single
rational map is not enough, which is why the grid is split.

In [10]:
from math import gamma as gamma_fn
A1n = -4/(7*np.sqrt(2)*np.pi**1.5)*gamma_fn(0.25)/gamma_fn(0.75); A2n = 8/15
A3n = -16408/3375*np.sqrt(2*np.pi)/np.pi**2*gamma_fn(0.75)/gamma_fn(0.25)
rs = 1e8; qstar = (4*ATF*rs/3)**0.25
three = 1000*(A1n/qstar+A2n/qstar**2+A3n/qstar**3)
v = [1000*kite(rs, (0.0,1.0,4.0,None), (16,16,40), nnu=44, N=150_000, seed=90+i) for i in range(5)]
m, se = np.mean(v), np.std(v,ddof=1)/np.sqrt(5)
note(f'r_s = 1e8 (q_* = {qstar:.1f}) against the three-term expansion', abs(m-three) < 3*se,
     f'{m:7.3f} +- {se:5.3f} mRy   three-term {three:7.3f}   {(m-three)/se:+.1f} sigma')

OK   r_s = 1e8 (q_* = 97.0) against the three-term expansion   -1.953 +- 0.130 mRy   three-term  -2.158   +1.6 sigma


## 8. The remainder claim of PRB Appendix C

The cross sector is the only part of the composite whose integrated form is not exact by
scaling, so it carries the remainder. The claim
$E_C=A_2/Q^2+A_{3,\rm cross}/Q^3+O(Q^{-5})$ is tested with the subtraction performed inside
the integrand, which avoids cancelling two large numbers. If the claim holds,
$Q^2\,[\,Q^3(E_C-A_2/Q^2)-A_{3,\rm cross}\,]$ approaches a constant; a $Q^{-4}$ term would
make it grow linearly in $Q$. Three values of $Q$ test consistency over a finite range; they do
not prove the asymptotic bound, which is argued in PRB Appendix C.

$C(q)$ is written here as $\int\mathrm d^3e\,V(|\bm e-q\hat{\bm z}|)/e^2$ with the support
kink split, which removes the $1/|\bm d+q\hat{\bm z}|^2$ singularity of the naive form.

In [11]:
def Cq(qv, ne=170, nc=170):
    # the integrand vanishes unless |e - q| < 2, so the radial nodes cover [max(0,q-2), q+2] only
    e,we = gl(ne,max(0.0,qv-2),2+qv); tot = 0.0
    c0,w0 = gl(nc,-1.0,1.0)
    for ei,wi in zip(e,we):
        ck = (ei**2+qv**2-4)/(2*ei*qv)
        if -1 < ck < 1:
            ss = 0.0
            for lo,hi in [(-1.0,ck),(ck,1.0)]:
                cc,wc = gl(nc//2,lo,hi)
                ss += (wc*Vov(np.sqrt(ei**2+qv**2-2*ei*qv*cc))).sum()
        else:
            ss = (w0*Vov(np.sqrt(ei**2+qv**2-2*ei*qv*c0))).sum()
        tot += wi*2*np.pi*ss
    return tot
rec('C(0) = I_0 = 4 pi^2  [independent representation]', 4*np.pi*float(mp.quad(Vcl,[0,2])), 4*float(pi)**2, tol=1e-9)
def Hx(xv):
    g = xv**-4
    if g < 1e-6: return xv**-2*(1-g/3+3*g**2/16-g**3/8)   # Taylor series of the closed form
    if g > 1e6:  return 4/3-2*xv**4
    return xv**-2*4/(3*g**2)*((1+g)**1.5-3*np.sqrt(1+g)+2)
A3cross = float(-mp.mpf(32)/15*CG)
tx,wx = gl(240,0.0,1.0); xs = tx/(1-tx); jxs = wx/(1-tx)**2
print('   Q        Q^3 (E_C - A2/Q^2)      A_{3,cross}        difference x Q^2')
vals = []
for Q in [8.0, 16.0, 32.0]:
    val = 8*np.pi*float(A0)*Q**2*sum(jv*Cq(Q*xv)*(Hx(xv)-4/3) for xv,jv in zip(xs,jxs))
    vals.append((val-A3cross)*Q**2)
    print(f'  {Q:<9}{val: .8f}        {A3cross: .8f}      {(val-A3cross)*Q**2: .5f}')
spread = max(vals)-min(vals)
note('finite-range consistency with an O(Q^-5) remainder: Q^2 x difference at Q = 8, 16, 32 stays within 25% of its mean',
     spread < 0.25*abs(np.mean(vals)), f'values {[round(v,4) for v in vals]}, spread {spread:.4f}')

OK   C(0) = I_0 = 4 pi^2  [independent representation]           39.4784176044  vs    39.4784176044  rel 0.0
   Q        Q^3 (E_C - A2/Q^2)      A_{3,cross}        difference x Q^2
  8.0      -0.18446413        -0.18312668      -0.08560
  16.0     -0.18347113        -0.18312668      -0.08818
  32.0     -0.18321344        -0.18312668      -0.08884
OK   finite-range consistency with an O(Q^-5) remainder: Q^2 x difference at Q = 8, 16, 32 stays within 25% of its mean values [np.float64(-0.0856), np.float64(-0.0882), np.float64(-0.0888)], spread 0.0032


## 9. Transcription of the published comparison values

PRB Table II quotes the $\zeta=0$ column of Table IX of
Benites, Rosado and Manousakis, Phys. Rev. B **110**, 195151 (2024). `brm_table_ix_zeta0.json` records
that column as extracted from the text layer of the paper by `transcribe_brm_table_ix.py`; the
cell compares it with `production/reference_brm.json`, which the production tables and figures
read. Both are read from file, so a transcription error in either appears as a mismatch.

In [12]:
import json, pathlib
_here = pathlib.Path('.').resolve()
_root = _here if (_here / 'production').exists() else _here.parent
BRM_TABLE_IX = {r[0]: (r[1], r[2]) for r in json.loads((_root / 'independent_audit' / 'brm_table_ix_zeta0.json').read_text())['rows']}
_ref = json.loads((_root / 'production' / 'reference_brm.json').read_text())
USED_BY_MANUSCRIPT = {r[0]: (r[1], r[2]) for key in ('tableI', 'tableII') for r in _ref[key] if r[1] is not None}
bad = [k for k in USED_BY_MANUSCRIPT if k not in BRM_TABLE_IX or any(abs(a - b) > 1e-9 for a, b in zip(BRM_TABLE_IX[k], USED_BY_MANUSCRIPT[k]))]
note(f'all {len(USED_BY_MANUSCRIPT)} published entries used by the manuscript match the transcribed Table IX',
     not bad, f'mismatches: {bad}')


OK   all 15 published entries used by the manuscript match the transcribed Table IX mismatches: []


## Summary

In [13]:
n = len(LOG); ok = sum(LOG)
print(f'{ok} of {n} independent items agree with the manuscripts; {n-ok} disagree.')
print(f'total runtime {time.perf_counter()-T0:.0f} s')
print()
print('Scope: this notebook re-derives quantitative content. It is not a proof check;')
print('the analytic arguments are in the manuscripts and their appendices.')
if n - ok:
    raise AssertionError(f'{n - ok} item(s) disagree; see the lines marked above')

70 of 70 independent items agree with the manuscripts; 0 disagree.
total runtime 513 s

Scope: this notebook re-derives quantitative content. It is not a proof check;
the analytic arguments are in the manuscripts and their appendices.
